# Random Forest Training

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
import os
import joblib
import json

## Loading Data

In [2]:
X_train=pd.read_csv("X_train.csv")
Y_train=pd.read_csv("Y_train.csv")
X_test=pd.read_csv("X_test.csv")
Y_test=pd.read_csv("Y_test.csv")

## Training

In [3]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [4]:
print("\nTraining Random Forest...")

rf_model.fit(X_train, Y_train)

print("Training completed.")


Training Random Forest...


C:\Users\ali\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Training completed.


In [5]:
Y_pred = rf_model.predict(X_test)
Y_prob = rf_model.predict_proba(X_test)[:, 1]

## Model Evaluation

In [6]:
print("\nClassification Report:")
print(classification_report(
    Y_test,
    Y_pred,
    target_names=["Benign", "Non-Benign"]
))

print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("Precision:", precision_score(Y_test, Y_pred))
print("Recall:", recall_score(Y_test, Y_pred))
print("F1 Score:", f1_score(Y_test, Y_pred))
print("ROC-AUC:", roc_auc_score(Y_test, Y_prob))
print("PR-AUC:", average_precision_score(Y_test, Y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, Y_pred))


Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     84976
  Non-Benign       0.89      0.78      0.83       142

    accuracy                           1.00     85118
   macro avg       0.94      0.89      0.92     85118
weighted avg       1.00      1.00      1.00     85118

Accuracy: 0.9994713221645245
Precision: 0.888
Recall: 0.7816901408450704
F1 Score: 0.8314606741573034
ROC-AUC: 0.9533737031963954
PR-AUC: 0.8216947450669224

Confusion Matrix:
[[84962    14]
 [   31   111]]


## Saving Model

In [12]:
os.makedirs("models", exist_ok=True)

joblib.dump(
    rf_model,
    "models/random_forest_model.pkl"
)

['models/random_forest_model.pkl']

## Saving Metrics

In [13]:
os.makedirs("random_forest", exist_ok=True)
metrics = {
    "accuracy": accuracy_score(Y_test, Y_pred),
    "precision": precision_score(Y_test, Y_pred),
    "recall": recall_score(Y_test, Y_pred),
    "f1_score": f1_score(Y_test, Y_pred),
    "roc_auc": roc_auc_score(Y_test, Y_prob),
    "pr_auc": average_precision_score(Y_test, Y_prob)
}

with open("random_forest/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Metrics saved.")

Metrics saved.


In [14]:
cm = confusion_matrix(Y_test, Y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual_Positive", "Actual_Negative"],
    columns=["Predicted_Positive", "Predicted_Negative"]
)

cm_df.to_csv(
    "random_forest/confusion_matrix.csv"
)

print("Confusion matrix saved.")

Confusion matrix saved.


In [15]:
report = classification_report(
    Y_test,
    Y_pred,
    target_names=["Benign", "Non-Benign"]
)

with open("random_forest/classification_report.txt", "w") as f:
    f.write(report)

print("Classification report saved.")

Classification report saved.


## Saving Predictions

In [17]:
predictions = pd.DataFrame({
    "actual": Y_test.values.ravel(),
    "prediction": Y_pred,
    "probability_non_benign": Y_prob
})

predictions.to_csv(
    "random_forest/predictions.csv",
    index=False
)